Contributors: Thomas Asikis, Lucas Bottcher
## Imports

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../')
import torch
from neural_control.extended_controllers import NNCDIController
from neural_control.extended_dynamics import SequentialDualSourcing
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from plotly import graph_objects as go
from sourcing_models.utilities import sample_trajectories_capped_dual_index, \
sample_trajectories_dual_index, sample_trajectories_single_index, sample_trajectories_tailored_base_surge
import time


from neural_control.experiments.helpers import load_experiment_conf

import matplotlib.pyplot as plt
from matplotlib import rcParams
#import seaborn as sns

# customized settings
params = {  # 'backend': 'ps',
    'font.family': 'serif',
    'font.serif': 'Latin Modern Roman',
    'font.size': 10,
    'axes.labelsize': 'medium',
    'axes.titlesize': 'medium',
    'legend.fontsize': 'medium',
    'xtick.labelsize': 'small',
    'ytick.labelsize': 'small',
    'savefig.dpi': 150,
    'text.usetex': True}
# tell matplotlib about your params
rcParams.update(params)

## Sourcing Problem Parameters

In [2]:
sourcing_parameters_path = '../../sourcing_models/recursion_input_files/low_service_fc1_lr=2_b=85_h=15_u04_fe=100_fr=50.in'
experiment_data = load_experiment_conf(sourcing_parameters_path)
total_time = 100
torch.set_default_tensor_type('torch.cuda.FloatTensor')

In [4]:
sourcing_parameters = dict(ce=int(experiment_data.c_e),
                           cr=int(experiment_data.c_r),
                           le=int(experiment_data.l_e),
                           lr=int(experiment_data.l_r),
                           h=int(experiment_data.h),
                           b=int(experiment_data.b),
                           T=total_time,
                           fe = 0,#int(experiment_data.f_e),
                           fr = 0,#int(experiment_data.f_r)

                          )

In [3]:
sourcing_parameters = dict(ce=20,
                           cr=0,
                           le=0,
                           lr=2,
                           h=5,
                           b=495,
                           T=200,
                           fe=100,
                           fr=50
                          )

In [4]:
sourcing_parameters

{'ce': 20,
 'cr': 0,
 'le': 0,
 'lr': 2,
 'h': 5,
 'b': 495,
 'T': 200,
 'fe': 100,
 'fr': 50}

## NNC Initialization
Here we initialize an NNC model and set  its hyperparameters for training.

In [5]:
### We use celu non-linearities for the input layer, hidden layers, and output layer

fcc = NNCDIController(lr=sourcing_parameters['lr'], 

                                        ) # controller neural network object


## Dynamics Initialization
To further optimize NNC (''fine tuning''), we now apply gradient propagation to the inventory evolution equation:

$I_t = I_{t-1} + q^{\rm e}_{t-l_{\rm e}} + q^{\rm r}_{t-l_{\rm r}} - D_t$

where $I_0, q^{\rm e}_{t},q^{\rm r}_{t}$ are NNC outputs.

In [6]:
dsd = SequentialDualSourcing(fcc, I_0=6, **sourcing_parameters) # Dual Sourcing Dynamics object

We preserve the best performing model in training, assuming that the neural network generalizes well during training and that our training sample is large enough.

In [7]:
best_loss = [np.infty]
best_model = [None]

In [8]:
all_training_costs = []

We observe that we have a neural network that outputs $q^{\rm r}_{t-l_{\rm r}}, q^{\rm e}_{t-l_{\rm e}}$ given the input described above.
The initial inventory $I_0$ is independent of inputs and can be learned separately (in a way that is similar to learning a bias term of a neural network).
We use separate optimizers to learn order quantities and initial inventory.
This may create learning variance, as one optimizer may overfit on a value that is directly changed by another optimizer. 
To reduce the learning variance, we let the order optimizer operate on 7 out of 10 learning epochs and the inventory optimizer on the remaining epochs.

In [16]:
optimizer = torch.optim.RMSprop(#[dsd.I_0],
                                list(fcc.parameters()), 
                                lr=3*1e-3
                               )
optimizer2 = torch.optim.RMSprop([dsd.I_0],
                                #list(fcc.parameters()), 
                                lr=1e-3
                               )

We now fine tune the model after imitation, in order to generalize and hopefully outperform the method used to train the model.
To do so, we repetively train and fine tune the parameters of 128 trajectories generated by setting a specific random seed. We do not resample new trajectories, as we want the network to overfit on the process. Sampling new samples could increase the input variance, and could potentially slow/prohibit convergence.

In [14]:
fcc.sf

Parameter containing:
tensor([5.0048], requires_grad=True)

In [15]:
fcc.ss

Parameter containing:
tensor([13.9975], requires_grad=True)

In [12]:
fcc.qbars

Parameter containing:
tensor([3.6788], requires_grad=True)

In [17]:
fine_tuning_iterations = 3000
minibatch_size = 256
random_seed = 6

t0 = time.time()
for it in range(fine_tuning_iterations):
    optimizer.zero_grad()
    optimizer2.zero_grad()
    def closure():
        dsd.reset(minibatch_size, seed=random_seed+1)
        nn_context = {}
        total_costs = 0
        for i in range(dsd.T):
            current_costs, demands, current_inventories, qr, qra, qe, qea, nn_context = dsd.simulate()
            total_costs = current_costs.mean() + total_costs
        all_training_costs.append(float(total_costs)/dsd.T)
        total_costs.backward()
        if it % 20 == 0:
            print(total_costs/dsd.T)
        if total_costs < best_loss[0]:
            best_loss[0] = total_costs.detach().cpu().item()
            best_model[0] = deepcopy(fcc.state_dict())
        return total_costs
    #if it > 1 and all_training_costs[-1] <= 26.4:
    #        break
    if it % 10 > 6:
        optimizer2.step(closure)
    else:
        optimizer.step(closure)
t1 = time.time()
total = t1-t0
print(total)

tensor(78.5999, grad_fn=<DivBackward0>)
tensor(78.5999, grad_fn=<DivBackward0>)
tensor(82.1153, grad_fn=<DivBackward0>)
tensor(78.5999, grad_fn=<DivBackward0>)
tensor(82.1153, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(76.3920, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(76.3920, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(76.3920, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(76.3920, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)
tensor(76.3920, grad_fn=<DivBackward0>)
tensor(75.9391, grad_fn=<DivBackward0>)


In [16]:
fcc.ss

Parameter containing:
tensor([11.0057], requires_grad=True)

In [17]:
fcc.sf

Parameter containing:
tensor([0.0029], requires_grad=True)

In [18]:
fcc.qbars

Parameter containing:
tensor([13.1695], requires_grad=True)

In [19]:
best_loss[0]/dsd.T

59.702734375

In [18]:
sourcing_parameters

{'ce': 5,
 'cr': 0,
 'le': 0,
 'lr': 2,
 'h': 15,
 'b': 85,
 'T': 100,
 'fe': 0,
 'fr': 0}

train one nn for lr=2,3,4

Takes several thousand epochs to train one initial NN nicely, then finetuning brings it 1-5% closer to DP
Fewer epochs for fine-tuning


transfer learn for new parameters except lr


lower lr and increase learning rate close to dp and then increase again.

Start from 10^-3 learning rate
minibatch size: 256 - 1024 work relative well.
for lr > 2 use T=150, 200, longer lrs need bigger lrs

Nice initial inventory level, too small and too large initial inventories may bias trainning. 
Between 4 and 8 seems to work very well.

In [19]:
fine_tuning_iterations = 3000
minibatch_size = 256
random_seed = 500

t0 = time.time()
dsd.reset(minibatch_size, seed=random_seed+1)
nn_context = {}
total_costs = 0
for i in range(dsd.T):
    current_costs, demands, current_inventories, qr, qra, qe, qea, nn_context = dsd.simulate()
    total_costs = current_costs.mean() + total_costs

print(total_costs/dsd.T)

t1 = time.time()
total = t1-t0
print(total)

tensor(59.9131, grad_fn=<DivBackward0>)
0.07316803932189941
